# w1-hab · Привычки и здоровье

Скелет анализа. Запускать сверху вниз после заполнения `data/PASSPORT.md`
и проверки `sha256`.

**Вопрос:** связаны ли питание, сон, стресс и активность с бинарным исходом?

**Границы:** данные наблюдательные — только ассоциации и гипотезы.

In [ ]:
from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd

ROOT = Path.cwd()
RAW = ROOT / "data" / "raw"
OUT = ROOT / "out"
OUT.mkdir(exist_ok=True)

SEED = 42
np.random.seed(SEED)
print(ROOT)

## 1. Паспорт источника

Проверяем ссылку, лицензию и sha256 **до** любой аналитики.

In [ ]:
passport = (ROOT / "data" / "PASSPORT.md").read_text(encoding="utf-8")
print(passport[:1500])

In [ ]:
def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

files = sorted(p for p in RAW.glob("*") if p.is_file() and p.name != ".gitkeep")
[(p.name, sha256(p)) for p in files]

## 2. Загрузка и блоки признаков

Колонки распределяются по блокам до моделирования; target задаётся явно.

In [ ]:
# TODO: заменить на фактический формат (csv/parquet/xlsx)
# df = pd.read_csv(files[0])

FEATURE_BLOCKS = {
    "nutrition": [],
    "sleep": [],
    "stress": [],
    "activity": [],
    "control": [],
}
TARGET = None  # имя бинарной колонки из spec.md
FEATURE_BLOCKS, TARGET

## 3. Очистка (без заполнения средним)

Дубликаты, типы, невозможные значения, пометка выбросов, complete-case.

In [ ]:
def clean(df: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    report = {"rows_in": len(df)}
    df = df.drop_duplicates().copy()
    report["rows_after_dedup"] = len(df)
    report["missing_share"] = df.isna().mean().round(3).to_dict()
    model_cols = [c for cols in FEATURE_BLOCKS.values() for c in cols] + [TARGET]
    report["rows_complete_case"] = int(df[model_cols].dropna().shape[0])
    # средним/медианой НЕ заполняем
    return df, report

# df, clean_report = clean(df)

## 4. Baseline и логистическая регрессия

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, brier_score_loss, log_loss,
                             roc_auc_score)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# X = df[model_cols without TARGET]
# y = df[TARGET]
# X_train, X_test, y_train, y_test = train_test_split(
#     X, y, test_size=0.25, stratify=y, random_state=SEED)

## 5. Permutation importance и 95% CI

1000 перестановок на test; для коэффициентов — bootstrap.

In [ ]:
from sklearn.inspection import permutation_importance

# baseline = DummyClassifier(strategy="prior").fit(X_train, y_train)
# model = pipeline.fit(X_train, y_train)
# perm = permutation_importance(model, X_test, y_test, scoring="roc_auc",
#                               n_repeats=1000, random_state=SEED)

## 6. Сохранение результатов

`out/metrics.json`, `out/factors.csv`, затем `report/report.html`.

In [ ]:
# (OUT / "metrics.json").write_text(
#     json.dumps(metrics, ensure_ascii=False, indent=2), encoding="utf-8")
# factors.to_csv(OUT / "factors.csv", index=False)
print("TODO: сохранить out/metrics.json и out/factors.csv")